<a href="https://colab.research.google.com/github/Areej973/Prompt-Chaining-and-Iterative-Generation-for-Story-Writing/blob/main/Prompt_Chaining_and_Iterative_Generation_for_Story_Writing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Chaining and Iterative Generation for Story Writing


This notebook demonstrates how to write a story using two  powerful tools: prompt chaining and iterative generation. These can be used to tackle complex tasks that are difficult or impossible to complete in a single step.

**Prompt chaining** involves breaking down a larger task into smaller, interconnected prompts. The output of each prompt then becomes the input for the next, guiding the language model through the process step-by-step. This approach offers several benefits:

*   Improved accuracy: Smaller, focused prompts can lead to better results from the language model.
*   Debugging: It's easier to identify where things go wrong within the chain, allowing for targeted adjustments and improvements.
*   Complex tasks: By breaking down intricate problems into manageable steps, prompt chaining enables the language model to tackle more complex tasks.

**Iterative generation** refers to the process of building the desired output iteratively. In this case, you will use it to write a story that is longer than what a single generation window allows. Iterative generation offers several benefits:

*   Longer outputs: It allows for the creation of longer and more detailed outputs, exceeding the limitations of a single generation window.
*   Flexibility: You can adjust and refine the output at each iteration, ensuring the story develops in the desired direction.
*   Human-in-the-loop control: You can provide feedback and guidance at each step, ensuring the story aligns with your creative vision.

By combining these techniques, you can create a compelling and well-structured story, piece by piece, while maintaining control over the creative process.



## Setup

In [1]:
!pip install -q -U "google-genai<=1.40.0,>=1.21.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.1/245.1 kB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 1.40.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 1.40.0 which is incompatible.


To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) quickstart for an example.

**Steps to get Gemini API access / API key**



*  Sign in / use your Google account
*   Go to Google AI Studio (AI for Developers)
You can obtain the Gemini API key from Google AI Studio.



* Import or create a Google Cloud project
*   If you don’t already have a project, you will need to create one (or import an existing one) into AI Studio so the key is associated with a project.



**Generate the API key**
*   In AI Studio’s “API Keys” section, you can create a new API key. You might need to accept terms of service, choose project, specify permissions, etc.
*  Enable the Gemini / Generative AI / relevant APIs in Google Cloud
*   Enable the Gemini / Generative AI / relevant APIs in Google Cloud

*  If you use it via Google Cloud / Vertex AI, you may need to enable the relevant APIs in the Google Cloud Console (for example, enabling “Generative AI” or “Gemini” APIs).



*  Use / configure the API key in your code or environment
*  You can set environment variables like GEMINI_API_KEY or GOOGLE_API_KEY.


*   Or supply the key explicitly in your API client / SDK initialization.


*  If using REST, include the key in headers (e.g. x-goog-api-key) as required.


In [3]:
from google import genai
from google.genai import types
from google.colab import userdata
from pprint import pprint

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
client=genai.Client(api_key=GOOGLE_API_KEY)

MODEL_ID="gemini-2.5-flash-lite" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash-lite-preview-09-2025", "gemini-2.5-flash", "gemini-2.5-flash-preview-09-2025", "gemini-2.5-pro"] {"allow-input":true, isTemplate: true}

## Prompts: Guiding the language model

You will use a series of interconnected prompts to guide the language model through the process of writing a story. These prompts will cover the story's premise, outline, and starting point, ultimately leading to the creation of a complete narrative.

It's important to carefully craft these prompts to provide clear instructions and relevant information to the language model. This will help the model generate high-quality content that aligns with your creative vision.

### Prompt chain for story writing

This section contains the prompts that will guide the language model through the story writing process. These prompts are designed to be chained together, with the output of one prompt feeding into the next.

Each prompt includes a **persona statement**, which helps the language model understand its role and generate more relevant and accurate content. In this case, the persona statement is: `You are an award-winning science fiction author with a penchant for expansive, intricately woven stories. Your ultimate goal is to write the next award winning sci-fi novel.`

Since the persona statement and writing guidelines appear in multiple prompts, f-string variables are used to add them to the prompts.

Additionally, the prompts use **placeholders** to insert the results of previous prompts using `.format()`. This allows us to build the story step-by-step, incorporating the outputs generated by the model at each stage. These are normally denoted by `{}`, but since you are also using f-string variables they are escaped as `{{}}`.

Here's a breakdown of the prompt chain:

1.  **Premise prompt**: This prompt asks the model to generate a single-sentence premise for a sci-fi story featuring cats.
1.  **Outline prompt**: This prompt provides the generated premise to the model and asks it to create a plot outline for the story.
1.  **Starting prompt**: This prompt provides both the premise and the outline to the model and asks it to begin writing the story. It also includes instructions to write a detailed and lengthy opening section.

By chaining these prompts together, it guides the language model through the process of creating a well-structured and engaging story.

#### Example

Let's say the model generates the following premise:

> In a future where humanity has achieved interstellar travel, a group of genetically enhanced cats embarks on a perilous mission to save the galaxy from a sinister alien threat.

This premise would then be inserted into the outline prompt:

> You are an award-winning science fiction author with a penchant for expansive,
intricately woven stories. Your ultimate goal is to write the next award winning
sci-fi novel.
>
> You have a gripping premise in mind:
>
> **In a future where humanity has achieved interstellar travel, a group of genetically enhanced cats embarks on a perilous mission to save the galaxy from a sinister alien threat.**
>
> Write an outline for the plot of your story.


The model would then generate an outline based on this premise, which would then be used in the starting prompt to begin writing the story itself.

This process continues iteratively, with the model generating additional content based on the previous outputs, until the story is complete.

In [4]:
persona = '''\
You are an award-winning science fiction author with a penchant for expansive,
intricately woven stories. Your ultimate goal is to write the next award winning
sci-fi novel.'''

guidelines = '''\
Writing Guidelines

Delve deeper. Lose yourself in the world you're building. Unleash vivid
descriptions to paint the scenes in your reader's mind. Develop your
characters—let their motivations, fears, and complexities unfold naturally.
Weave in the threads of your outline, but don't feel constrained by it. Allow
your story to surprise you as you write. Use rich imagery, sensory details, and
evocative language to bring the setting, characters, and events to life.
Introduce elements subtly that can blossom into complex subplots, relationships,
or worldbuilding details later in the story. Keep things intriguing but not
fully resolved. Avoid boxing the story into a corner too early. Plant the seeds
of subplots or potential character arc shifts that can be expanded later.

Remember, your main goal is to write as much as you can. If you get through
the story too fast, that is bad. Expand, never summarize.
'''

premise_prompt = f'''\
{persona}

Write a single sentence premise for a sci-fi story featuring cats.'''

outline_prompt = f'''\
{persona}

You have a gripping premise in mind:

{{premise}}

Write an outline for the plot of your story.'''

starting_prompt = f'''\
{persona}

You have a gripping premise in mind:

{{premise}}

Your imagination has crafted a rich narrative outline:

{{outline}}

First, silently review the outline and the premise. Consider how to start the
story.

Start to write the very beginning of the story. You are not expected to finish
the whole story now. Your writing should be detailed enough that you are only
scratching the surface of the first bullet of your outline. Try to write AT
MINIMUM 1000 WORDS and MAXIMUM 2000 WORDS.

{guidelines}'''

### Continuation prompt: Building the story

Once the language model has generated the beginning of the story, you can use a **continuation prompt** to iteratively expand the narrative. This prompt is similar to the starting prompt, but with two key differences:

1.  **Instruction to signal completion**: An instruction was added for the model to write `IAMDONE` when it believes the story is finished. This serves as a signal for us to stop generating additional content.
1.  **Work in progress**: The language in the prompt is adjusted to reflect that the story is already in progress, rather than starting from scratch.

The continuation prompt provides the model with the story's premise, outline, and the existing draft. It then instructs the model to continue writing the story in detail.

This iterative process allows us to build a longer and more complex story than what would be possible in a single generation call. You can continue feeding the existing draft back into the continuation prompt until the model signals that the story is complete by writing `IAMDONE`.

**Note**: The `IAMDONE` signal is simply a convenient way to identify the story's end in this example. In other applications of iterative generation, different methods might be used to determine when the desired output is complete.

In [5]:
continuation_prompt = f'''\
{persona}

You have a gripping premise in mind:

{{premise}}

Your imagination has crafted a rich narrative outline:

{{outline}}

You've begun to immerse yourself in this world, and the words are flowing.
Here's what you've written so far:

{{story_text}}

=====

First, silently review the outline and story so far. Identify what the single
next part of your outline you should write.

Your task is to continue where you left off and write the next part of the story.
You are not expected to finish the whole story now. Your writing should be
detailed enough that you are only scratching the surface of the next part of
your outline. Try to write AT MINIMUM 1000 WORDS. However, only once the story
is COMPLETELY finished, write IAMDONE. Remember, do NOT write a whole chapter
right now.

{guidelines}'''

## Writing time!

### Generate and print the premise

In [6]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=premise_prompt,
    )
premise = response.text
print(premise)

When humanity's interstellar probes discover a derelict Dyson Sphere built by an impossibly ancient feline civilization, a lone xenolinguist must decipher their silent, star-spanning legacy to prevent the Sphere's awakening from triggering a galactic cataclysm.


### Generate and print the outline

In [7]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=outline_prompt.format(premise=premise),
    )
outline = response.text
print(outline)

## Project: The Whispering Stars of Ailuros

**Logline:** When humanity's interstellar probes discover a derelict Dyson Sphere built by an impossibly ancient feline civilization, a lone xenolinguist must decipher their silent, star-spanning legacy to prevent the Sphere's awakening from triggering a galactic cataclysm.

**Genre:** Hard Sci-Fi, Space Opera, Mystery, Philosophical Sci-Fi

**Target Audience:** Readers of Arthur C. Clarke, Alastair Reynolds, Ted Chiang, and Liu Cixin.

**Award Potential:** Hugely ambitious, with scope for intricate world-building, profound thematic exploration, and a gripping, high-stakes plot. Aims for Hugo, Nebula, and Locus awards.

---

## Novel Outline: The Whispering Stars of Ailuros

**Part I: The Echo of a Lost Sun**

*   **Chapter 1: The Signal from the Void**
    *   Introduction to humanity's current state: a nascent interstellar civilization, slowly expanding, with limited understanding of the cosmos.
    *   The discovery of anomalous energy re

### Generate the start of the story

In [8]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=starting_prompt.format(premise=premise, outline=outline),
    )
starting_draft = response.text
pprint(starting_draft)

('The hum of the observatory was a lullaby Aris Thorne had grown to loathe. It '
 'was a sterile, incessant drone that echoed the emptiness he felt within the '
 'grand, vaulted dome, and more importantly, within his own stagnant career. '
 'Here, perched on a solitary peak that clawed at the thin Martian atmosphere, '
 'the universe felt both impossibly vast and suffocatingly small. His '
 'colleagues, back on a more populated, less dust-choked Earth, called it '
 'exile. Aris, in the quiet, biting honesty of his own mind, called it a '
 'gilded cage. His reputation, once a shimmering promise of groundbreaking '
 'discovery, had curdled into a cautionary tale. The "Thorne Anomaly," a '
 'theoretical framework for interspecies communication based on sub-quantum '
 'entanglement patterns, had been soundly, embarrassingly, debunked. The '
 'scientific community, a fickle beast prone to chasing the next shiny '
 'hypothesis, had turned its back.\n'
 '\n'
 'He traced a finger across the co

### Generate the continuation of the story and check progress

In [9]:
draft = starting_draft

response = client.models.generate_content(
    model=MODEL_ID,
    contents=continuation_prompt.format(premise=premise, outline=outline, story_text=draft),
    )

continuation = response.text
pprint(continuation)

('The stark, crystalline beauty of the void, once a source of Aris’s quiet '
 'contemplation, now felt like a prelude to an overwhelming symphony of the '
 'unknown. The _Odyssey_ drifted closer, her ventral lights casting tentative '
 'beams onto the immense, cyclopean structures of the Dyson Sphere. It was a '
 'sight that defied easy categorization, a monument to an intelligence so '
 "alien, so ancient, that it bordered on the sublime. The surface wasn't "
 'smooth, not in the way a machined surface would be. Instead, it was a '
 'landscape of impossible geometry, vast, obsidian planes punctuated by '
 'colossal, crystalline spires that seemed to drink the faint starlight. '
 'Intricate etchings, too fine to be mere decoration, crisscrossed every '
 'visible surface, forming patterns that hinted at a profound, yet '
 'undecipherable, significance.\n'
 '\n'
 '“Sensors are… struggling, Dr. Thorne,” Captain Eva Rostova’s voice, usually '
 'a steady anchor of authority, held a tremor o

### Let's finish writing the story.

After the iterative generation process is complete, the draft will contain the full story along with the `IAMDONE` signal at the end. The last part of this cell removes the `IAMDONE` signal and trims any unnecessary whitespace from the beginning and end of the draft, resulting in the final draft.

In [10]:
# Add the continuation to the initial draft, keep building the story until 'IAMDONE' is seen
draft = draft + '\n\n' + continuation

while 'IAMDONE' not in continuation:
  response = client.models.generate_content(
    model=MODEL_ID,
    contents=continuation_prompt.format(premise=premise, outline=outline, story_text=draft),
    )
  continuation = response.text
  draft = draft + '\n\n' + continuation

# Remove 'IAMDONE' and print the final story
final = draft.replace('IAMDONE', '').strip()
pprint(final)

('The hum of the observatory was a lullaby Aris Thorne had grown to loathe. It '
 'was a sterile, incessant drone that echoed the emptiness he felt within the '
 'grand, vaulted dome, and more importantly, within his own stagnant career. '
 'Here, perched on a solitary peak that clawed at the thin Martian atmosphere, '
 'the universe felt both impossibly vast and suffocatingly small. His '
 'colleagues, back on a more populated, less dust-choked Earth, called it '
 'exile. Aris, in the quiet, biting honesty of his own mind, called it a '
 'gilded cage. His reputation, once a shimmering promise of groundbreaking '
 'discovery, had curdled into a cautionary tale. The "Thorne Anomaly," a '
 'theoretical framework for interspecies communication based on sub-quantum '
 'entanglement patterns, had been soundly, embarrassingly, debunked. The '
 'scientific community, a fickle beast prone to chasing the next shiny '
 'hypothesis, had turned its back.\n'
 '\n'
 'He traced a finger across the co

Language models like Gemini process text in units called tokens. For Gemini models, each token is equivalent to about 4 characters.

`gemini-2.0-flash` has an output limit of 8192 tokens per generation call. This means that each individual prompt response cannot exceed this limit. By using iterative generation, you can create a story that is much longer than 8192 tokens by building it piece by piece.

Let's see how many tokens the final story is. Is it longer than 8192 tokens?

In [11]:
# Check the number of tokens in the final story
# gemini-2.0-flash output token limit is 8192
total_tokens=client.models.count_tokens(
    model=MODEL_ID,
    contents=final,
    )
print(f'Total tokens: {total_tokens.total_tokens}')

Total tokens: 15393
